In [1]:
import numpy as np
import pandas as pd

VARIANT = 1
np.random.seed(VARIANT)

base_temp = 9.5    # Варіант 1 
amplitude = 14.0   # Сезонна амплітуда
city = "Київ"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1)
        })

climate = pd.DataFrame(rows)
print(f"Розмірність DataFrame: {climate.shape}")
print("Перші 5 рядків набору даних:")
print(climate.head())

Розмірність DataFrame: (48, 4)
Перші 5 рядків набору даних:
  місто   рік  місяць  температура
0  Київ  2021       1         -2.9
1  Київ  2021       2         -3.2
2  Київ  2021       3          2.0
3  Київ  2021       4          8.4
4  Київ  2021       5         17.4


In [2]:
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"]).round(2)
print("Агреговані показники за роками:")
print(yearly_stats)

Агреговані показники за роками:
      mean  min   max
рік                  
2021  9.38 -4.7  25.2
2022  9.52 -4.8  23.5
2023  9.18 -3.6  22.8
2024  9.77 -5.6  23.3


In [3]:
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"]).round(2)
print("Агреговані показники за місяцями:")
print(monthly_stats)

max_std_month = monthly_stats["std"].idxmax()
max_std_val = monthly_stats.loc[max_std_month, "std"]
print(f"\nМісяць із найвищим розкидом (std): {max_std_month} (std = {max_std_val} °C)")

Агреговані показники за місяцями:
         mean   std
місяць             
1       -4.22  1.21
2       -2.98  0.40
3        3.05  1.02
4        8.90  0.87
5       16.55  0.57
6       20.72  1.18
7       23.58  1.13
8       21.90  1.09
9       16.15  0.66
10       9.38  0.85
11       2.98  0.95
12      -2.48  1.73

Місяць із найвищим розкидом (std): 12 (std = 1.73 °C)


In [4]:
pivot_climate = climate.pivot_table(
    index="місяць", 
    columns="рік", 
    values="температура", 
    aggfunc="mean"
)
print("Зведена таблиця (місяці в рядках, роки в стовпцях):")
print(pivot_climate)

Зведена таблиця (місяці в рядках, роки в стовпцях):
рік     2021  2022  2023  2024
місяць                        
1       -2.9  -4.8  -3.6  -5.6
2       -3.2  -3.0  -3.3  -2.4
3        2.0   3.6   2.4   4.2
4        8.4   8.4   8.6  10.2
5       17.4  16.3  16.2  16.3
6       19.3  20.7  22.2  20.7
7       25.2  23.5  22.8  22.8
8       20.9  22.2  21.2  23.3
9       16.8  15.4  15.8  16.6
10       9.3  10.6   8.7   8.9
11       4.0   3.4   1.8   2.7
12      -4.7  -2.1  -2.6  -0.5


In [5]:
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)

climate["тепліше_за_середнє"] = climate["температура"] > base_temp

ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print("Таблиця спряженості (сезон vs тепліше за середнє):")
print(ct)

Таблиця спряженості (сезон vs тепліше за середнє):
тепліше_за_середнє  False  True 
сезон                           
весна                   7      5
зима                   12      0
літо                    0     12
осінь                   7      5


In [6]:
try:
    direct_pivot = climate.pivot(index="місяць", columns="рік", values="температура")
    print("Виклик .pivot() пройшов успішно!")
    print(direct_pivot.head())
except Exception as e:
    print(f"Виникла помилка: {e}")

Виклик .pivot() пройшов успішно!
рік     2021  2022  2023  2024
місяць                        
1       -2.9  -4.8  -3.6  -5.6
2       -3.2  -3.0  -3.3  -2.4
3        2.0   3.6   2.4   4.2
4        8.4   8.4   8.6  10.2
5       17.4  16.3  16.2  16.3
